# Business Entity Resolution: Version 10 (fix the remaining mistakes, type by type)

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**.
Settings: **CPU** (no accelerator), **Internet On**. It runs on Kaggle's servers even if you close the tab.
Expected time: about 1.5 to 2 hours of training, then 5 to 7 hours of test prediction (under Kaggle's 12-hour limit).

Version 10 started from an error analysis: every mistake on 30k held-out training businesses was sorted by cause,
and the biggest causes were fixed one by one (checked on the same 30k businesses as v1, v7 and v8):

| Step | Val F0.5 |
|---|---|
| v8 features alone / v7 alone | 0.9673 / 0.9647 |
| v7 house-number + legal-form checks together with v8 features | 0.9741 |
| + word-by-word name checks (one changed word, initials such as "mc.com") | 0.9747 |
| + second model that looks at all candidates of a business together (v2's idea) | **0.9759** |
| + cleaning fixes (Indian-script words, state names such as Ohio/TG/தமிழ்நாடு, digit typos) | printed by this run |

The candidate list (`candidate_pairs.tsv`) keeps only candidates with a first-stage chance of at least 1%
(about 4 to 5 per business, at most 20). The organisers rank smaller candidate sets higher.

Cells: 1 install, 2 to 4 pipeline code, 5 settings, 6 **build training table**, 7 **train** (look for `FINAL: val F0.5`),
8 **predict** test and write both TSVs to `output/`, 9 official format check.

No external data or APIs: only the competition files. Licenses: LightGBM MIT, rapidfuzz MIT, jellyfish MIT, polars MIT.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm jellyfish xgboost catboost

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")

# =====================================================================
# v8 additions: richer pair features + an ensemble of gradient boosting models.
# Everything here works on top of the v1 pipeline (erpipe) and uses no external data.
# Libraries: jellyfish (MIT), xgboost (Apache-2.0), catboost (Apache-2.0), lightgbm (MIT).
# =====================================================================
import jellyfish
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
import xgboost as xgb
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------
# 1. Extra cleaned columns per record
#    name_w  : the name's words without legal words (Pvt, Ltd, LLC, SARL ...)
#    name_ph : the same words as sound codes (metaphone), so "Sharma" and "Sharmaa" give the same code
# ---------------------------------------------------------------------
def _extra_rows(names):
    out = []
    for nb in names:
        toks = nb.split()
        core = [t for t in toks if skel_tok(t) not in LEGAL_SK] or toks
        ph = [jellyfish.metaphone(t) if not t.isdigit() else t for t in core]
        out.append((" ".join(core), " ".join(p for p in ph if p)))
    return out

def prepare_extra(df):
    names = df.name_b.tolist(); step = 100_000
    with Pool(N_CPU) as pool:
        res = pool.map(_extra_rows, [names[i:i + step] for i in range(0, len(names), step)])
    res = [r for chunk in res for r in chunk]
    df["name_w"] = [r[0] for r in res]
    df["name_ph"] = [r[1] for r in res]
    return df

# ---------------------------------------------------------------------
# 2. Word-rarity ("IDF") weights, learned from the candidate pool itself.
#    A word shared by 50,000 businesses ("store", "road") says little; a rare word ("zanskar") says a lot.
# ---------------------------------------------------------------------
class Rarity:
    """Word vocabularies + rarity weights for one candidate pool (one source, one country)."""
    def __init__(self, B):
        self.parts = {}
        for col in ("name_w", "addr_n", "name_ph"):
            v = CountVectorizer(token_pattern=r"\S+", binary=True, dtype=np.float32)
            try:
                X = v.fit_transform(B[col].to_numpy())
            except ValueError:            # empty vocabulary
                self.parts[col] = None; continue
            df = np.asarray(X.sum(axis=0)).ravel()
            idf = (np.log((len(B) + 1) / (df + 1)) + 1).astype(np.float32)
            self.parts[col] = (v, idf, X.tocsr(), float(np.log(len(B) + 1) + 1))
        # full char 3-gram similarity of the name skeleton for EVERY pair (the shortlist only kept top-15)
        self.cv = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
        try:
            self.CB = self.cv.fit_transform(B.name_skc.to_numpy()).tocsr()
        except ValueError:
            self.cv = None

def _rowdot(XA, XB):
    return np.asarray(XA.multiply(XB).sum(axis=1)).ravel().astype(np.float32)

def _rowmax(X):
    if X.nnz == 0:
        return np.zeros(X.shape[0], np.float32)
    return np.asarray(X.max(axis=1).todense()).ravel().astype(np.float32)

def rarity_features(R, a, b, A, f, chunk=3_000_000):
    """Adds rarity-weighted similarities for (A row a, pool row b) pairs into DataFrame f."""
    for col, pre in (("name_w", "rn"), ("addr_n", "ra"), ("name_ph", "rp")):
        names = [f"{pre}_{x}" for x in ("wjacc", "cont_s1", "cont_c", "cos", "miss_s1", "miss_c", "share_max")]
        if R.parts[col] is None:
            for n in names: f[n] = np.float32(np.nan)
            continue
        v, idf, XB_all, max_idf = R.parts[col]
        texts = A[col].to_numpy()
        XA_all = v.transform(texts).tocsr()
        # words of the S1 record that never occur in the pool: maximally rare, and surely not shared
        n_unseen = np.array([len(set(t.split())) for t in texts], np.float32) - np.asarray(XA_all.sum(axis=1)).ravel()
        W = sp.diags(idf); W2 = sp.diags(idf * idf)
        XAw, XBw = (XA_all @ W).tocsr(), (XB_all @ W).tocsr()
        sA = np.asarray(XAw.sum(axis=1)).ravel() + n_unseen * max_idf
        sB = np.asarray(XBw.sum(axis=1)).ravel()
        nA = np.sqrt(np.asarray((XA_all @ W2).sum(axis=1)).ravel() + n_unseen * max_idf ** 2)
        nB = np.sqrt(np.asarray((XB_all @ W2).sum(axis=1)).ravel())
        res = {n: np.empty(len(a), np.float32) for n in names}
        for i in range(0, len(a), chunk):
            ai, bi = a[i:i + chunk], b[i:i + chunk]
            PA, PB = XAw[ai], XB_all[bi]
            S = PA.multiply(PB).tocsr()                               # shared words, with their weights
            inter = np.asarray(S.sum(axis=1)).ravel()
            inter2 = np.asarray(S.multiply(S).sum(axis=1)).ravel()
            sa, sb = sA[ai], sB[bi]
            with np.errstate(divide="ignore", invalid="ignore"):
                res[names[0]][i:i + chunk] = np.where(sa + sb - inter > 0, inter / (sa + sb - inter), np.nan)
                res[names[1]][i:i + chunk] = np.where(sa > 0, inter / sa, np.nan)
                res[names[2]][i:i + chunk] = np.where(sb > 0, inter / sb, np.nan)
                res[names[3]][i:i + chunk] = np.where(nA[ai] * nB[bi] > 0, inter2 / (nA[ai] * nB[bi]), np.nan)
            missA = _rowmax((PA - PA.multiply(PB > 0)).tocsr())        # rarest S1 word the candidate lacks
            res[names[4]][i:i + chunk] = np.maximum(missA, np.where(n_unseen[ai] > 0, max_idf, 0))
            PBw = XBw[bi]
            res[names[5]][i:i + chunk] = _rowmax((PBw - PBw.multiply(PA > 0)).tocsr())   # rarest candidate word S1 lacks
            res[names[6]][i:i + chunk] = _rowmax(S)                   # rarest word they share
        for n in names: f[n] = res[n]
    if R.cv is not None:
        CA = R.cv.transform(A.name_skc.to_numpy()).tocsr()
        out = np.empty(len(a), np.float32)
        for i in range(0, len(a), chunk):
            out[i:i + chunk] = _rowdot(CA[a[i:i + chunk]], R.CB[b[i:i + chunk]])
        f["nch_cos"] = out
    else:
        f["nch_cos"] = np.float32(np.nan)

def phonetic_features(a, b, A, B, f):
    x, y = A.name_ph.to_numpy()[a].tolist(), B.name_ph.to_numpy()[b].tolist()
    f["ph_tset"] = cpdist(x, y, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)
    f["ph_ratio"] = cpdist(x, y, scorer=fuzz.ratio, workers=N_CPU).astype(np.float32)
    f["ph_first_eq"] = np.array([(p.split(" ", 1)[0] == q.split(" ", 1)[0]) if p and q else -1
                                 for p, q in zip(x, y)], np.int8)
    xw, yw = A.name_w.to_numpy()[a].tolist(), B.name_w.to_numpy()[b].tolist()
    f["nw_tset"] = cpdist(xw, yw, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)

# ---------------------------------------------------------------------
# 3. Rank features: how does this candidate compare with the business's other candidates (same source)?
# ---------------------------------------------------------------------
RANK_ON = ["rn_cos", "ra_cos", "nch_cos", "ph_tset", "combo"]

def rank_features(a, f):
    f["combo"] = (f.rn_cos.fillna(0) + f.ra_cos.fillna(0) + f.nch_cos.fillna(0) + f.ph_tset / 100).astype(np.float32)
    g = pd.DataFrame({"a": a})
    for k in RANK_ON:
        g["v"] = f[k].fillna(-1).to_numpy()
        grp = g.groupby("a")["v"]
        mx = grp.transform("max").to_numpy()
        f[f"rk_{k}"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
        f[f"gap_{k}"] = (mx - g.v.to_numpy()).astype(np.float32)
    v = g.v.to_numpy()                      # g.v is "combo" now (last in RANK_ON)
    # runner-up value per business, found by sorting once
    order = np.lexsort((-v, a)); a_s, v_s = a[order], v[order]
    first = np.r_[True, a_s[1:] != a_s[:-1]]
    pos = np.arange(len(a_s)); start = np.maximum.accumulate(np.where(first, pos, 0))
    has2 = (start + 1 < len(a_s)) & (a_s[np.minimum(start + 1, len(a_s) - 1)] == a_s)
    sec_s = np.where(has2, v_s[np.minimum(start + 1, len(a_s) - 1)], -1)
    second = np.empty_like(v); second[order] = sec_s
    # top candidate: lead over the runner-up; others: distance behind the leader (negative)
    f["combo_lead"] = np.where(v >= mx, v - second, v - mx).astype(np.float32)
    f["combo_z"] = ((g.v - grp.transform("mean")) / (grp.transform("std").fillna(0) + 1e-3)).to_numpy(np.float32)
    f["n_close"] = g.assign(c=(mx - v) < 0.15).groupby("a")["c"].transform("sum").to_numpy(np.int16)

def features_v8(P, A, B, is_s3, R):
    """v1 features + v8 features for the shortlist pairs P of A (S1 batch) against pool B."""
    f = features(P, A, B, is_s3)
    a, b = P.a.to_numpy(), P.b.to_numpy()
    rarity_features(R, a, b, A, f)
    phonetic_features(a, b, A, B, f)
    rank_features(a, f)
    return f

# ---------------------------------------------------------------------
# 4. The ensemble: several boosting models, each gives a probability; the blend is their average.
# ---------------------------------------------------------------------
MODEL_SPECS = [
    ("lgb_a", "lgb", dict(learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=0)),
    ("lgb_b", "lgb", dict(learning_rate=0.05, num_leaves=127, min_child_samples=50, subsample=0.7, subsample_freq=1,
                          colsample_bytree=0.5, reg_lambda=5.0, random_state=1)),
    ("xgb", "xgb", dict(eta=0.08, max_depth=9, min_child_weight=5, subsample=0.8, colsample_bytree=0.7,
                        tree_method="hist", max_bin=256, eval_metric="logloss", objective="binary:logistic", seed=2)),
    ("cat", "cat", dict(learning_rate=0.12, depth=8, l2_leaf_reg=3, random_seed=3, border_count=128)),
]

def fit_model(kind, params, Xtr, ytr, Xva, yva, max_rounds, n_rounds=None):
    """Trains one model. With n_rounds=None it stops when the held-out loss stops improving."""
    es = n_rounds is None
    rounds = max_rounds if es else n_rounds
    if kind == "lgb":
        m = lgb.LGBMClassifier(n_estimators=rounds, n_jobs=N_CPU, verbose=-1, **params)
        m.fit(Xtr, ytr, **({"eval_set": [(Xva, yva)], "callbacks": [lgb.early_stopping(100, verbose=False)]} if es else {}))
        return m, (m.best_iteration_ if es else rounds)
    if kind == "xgb":
        dtr = xgb.QuantileDMatrix(Xtr, ytr, max_bin=params.get("max_bin", 256), nthread=N_CPU)
        ev = [(xgb.DMatrix(Xva, yva, nthread=N_CPU), "val")] if es else []
        m = xgb.train({**params, "nthread": N_CPU}, dtr, rounds, evals=ev,
                      early_stopping_rounds=100 if es else None, verbose_eval=False)
        del dtr; gc.collect()
        return m, (m.best_iteration + 1 if es else rounds)
    if kind == "cat":
        m = CatBoostClassifier(iterations=rounds, thread_count=N_CPU, verbose=False, allow_writing_files=False,
                               **({"od_type": "Iter", "od_wait": 100, "use_best_model": True} if es else {}), **params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva) if es else None)
        return m, (m.get_best_iteration() + 1 if es else rounds)

def predict_model(kind, m, X):
    if kind == "lgb":
        return m.predict(X, num_threads=N_CPU).astype(np.float32) if isinstance(m, lgb.Booster) \
            else m.predict_proba(X)[:, 1].astype(np.float32)
    if kind == "xgb":
        return m.predict(xgb.DMatrix(X, nthread=N_CPU)).astype(np.float32)
    return m.predict_proba(X)[:, 1].astype(np.float32)

def save_model(kind, m, path):
    if kind == "lgb": m.booster_.save_model(path)
    elif kind == "xgb": m.save_model(path)
    else: m.save_model(path)

def load_model(kind, path):
    if kind == "lgb": return lgb.Booster(model_file=path)
    if kind == "xgb":
        m = xgb.Booster(); m.load_model(path); m.set_param({"nthread": N_CPU}); return m
    m = CatBoostClassifier(); m.load_model(path); return m

MODEL_EXT = {"lgb": "txt", "xgb": "json", "cat": "cbm"}

# =====================================================================
# Version 7: exact "keys" (rule layer)
# A key is a short piece of text built from a record, e.g. "house number + street word + one word of the name".
# Two records that share a rare key are very likely the same business.
# We use keys in three ways:
#   1. extra shortlist: pairs sharing a key are added even if the TF-IDF shortlist missed them
#   2. extra features for LightGBM: which keys matched, and how common each key is
#   3. rules: key types that were (almost) always right on training data accept a pair directly
# Needs the functions from the v1 pipeline cell (prepare, skel_tok, log, N_CPU ...).
# =====================================================================
from hashlib import blake2b
import ctypes

def free_mem():
    """Free memory and hand it back to the system (so helper processes don't copy it)."""
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass

KEY_TYPES = ["name", "nameset", "name_num", "addr", "addr_tok", "tok2_num"]
KCAP = 30                    # keys shared by more than this many records on one side are too common to use

def _ktok(t):
    """Extra-strong spelling key for one word: skeleton, and c/k, w/v, j/z sound the same.
    ('healthcare' and the transliterated Hindi 'helthker' both become 'hltkr')."""
    return skel_tok(t).replace("c", "k").replace("j", "z")

def _h(s):
    return int.from_bytes(blake2b(s.encode(), digest_size=8).digest(), "little", signed=True)

def _keys_rows(args):
    rows, start = args
    ri, kt, kh = [], [], []
    for off, (country, name_sk, addr_n) in enumerate(rows):
        r = start + off
        toks = [_ktok(t) for t in name_sk.split() if t]
        toks = [t for t in toks if t]
        ks = []
        cat = "".join(toks)
        if len(cat) >= 4:
            ks.append((0, cat))
            ks.append((1, "".join(sorted(set(toks)))))
        at = addr_n.split()
        nums = []
        for i, t in enumerate(at):                          # every number in the address + the next word with letters
            d = re.sub(r"\D", "", t)
            if not d or len(nums) >= 3:
                continue
            d = d.lstrip("0") or "0"
            nxt = next((_ktok(x) for x in at[i + 1:i + 4] if re.search(r"[a-z]", x)), "")
            nums.append((d, nxt))
        big = [t for t in toks if len(t) >= 3 and not t.isdigit()][:4]
        for d, nxt in dict.fromkeys(nums):
            if len(cat) >= 4:
                ks.append((2, cat + "|" + d))
            if nxt:
                ks.append((3, d + "|" + nxt))
                for t in big:
                    ks.append((4, d + "|" + nxt + "|" + t))
            if len(big) >= 2:
                ks.append((5, big[0] + "|" + big[1] + "|" + d))
        for t, s in dict.fromkeys(ks):
            ri.append(r); kt.append(t); kh.append(_h(f"{country}|{t}|{s}"))
    return np.array(ri, np.int32), np.array(kt, np.int8), np.array(kh, np.int64)

def make_keys(D):
    """All keys of every record in a prepared table. Returns polars DataFrame(row, kt, kh, cnt)
    where cnt = how many records of this table share the key."""
    rows = list(zip(D.country.tolist(), D.name_sk.tolist(), D.addr_n.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_keys_rows, [(rows[i:i + step], i) for i in range(0, len(rows), step)])
    K = pl.DataFrame({"row": np.concatenate([r[0] for r in res]), "kt": np.concatenate([r[1] for r in res]),
                      "kh": np.concatenate([r[2] for r in res])})
    K = K.with_columns(pl.len().over("kh").cast(pl.Int32).alias("cnt"))
    return K

def key_pairs(KA, KB, a_rows=None):
    """Pairs (a, b) that share at least one key, with per key type: hit, how many S1 / candidates share it.
    KA: keys of Source 1 (row = row in the S1 table); KB: keys of the candidate table.
    a_rows: optional array of S1 rows to keep (a batch); output 'a' is then the position inside a_rows."""
    ka = KA.filter(pl.col("cnt") <= KCAP)
    if a_rows is not None:
        pos = pl.DataFrame({"row": np.asarray(a_rows, np.int32), "a": np.arange(len(a_rows), dtype=np.int32)})
        ka = ka.join(pos, on="row").drop("row")
    else:
        ka = ka.rename({"row": "a"})
    kb = KB.filter(pl.col("cnt") <= KCAP).rename({"row": "b", "cnt": "cb"})
    J = ka.rename({"cnt": "ca"}).join(kb.drop("kt"), on="kh")
    if J.height == 0:
        return None
    agg = []
    for t, nm in enumerate(KEY_TYPES):
        m = pl.col("kt") == t
        agg += [m.any().cast(pl.Int8).alias(f"k_{nm}"),
                pl.col("ca").filter(m).min().alias(f"k_{nm}_ca"),
                pl.col("cb").filter(m).min().alias(f"k_{nm}_cb")]
    agg += [(pl.col("kt") == 4).sum().cast(pl.Int16).alias("k_addr_tok_n"), pl.len().cast(pl.Int16).alias("k_n")]
    return J.group_by(["a", "b"]).agg(agg).to_pandas()

KEY_FEATS = [f"k_{nm}{s}" for nm in KEY_TYPES for s in ("", "_ca", "_cb")] + ["k_addr_tok_n", "k_n"]

def merge_key_pairs(P, KP, key_only_types=("name", "nameset", "name_num", "addr_tok", "tok2_num")):
    """Add key features to the TF-IDF shortlist P(a, b, sim_name, sim_addr), and add key-only pairs
    (pairs the TF-IDF shortlist missed) when a name-based key matched."""
    if KP is None or len(KP) == 0:
        for c in KEY_FEATS:
            P[c] = np.nan if c.endswith(("_ca", "_cb")) else 0
        P["from_tfidf"] = np.int8(1)
        return P
    KP = KP.copy()
    rare = [(KP[f"k_{t}"].to_numpy() == 1) & (KP[f"k_{t}_ca"].fillna(1e9).to_numpy() <= 1)
            & (KP[f"k_{t}_cb"].fillna(1e9).to_numpy() <= 3) for t in key_only_types]
    extra = KP[np.logical_or.reduce(rare)]                   # only rare keys may add pairs the shortlist missed
    P = P.merge(KP, on=["a", "b"], how="left")
    P["from_tfidf"] = np.int8(1)
    new = extra.merge(P[["a", "b"]], on=["a", "b"], how="left", indicator=True)
    new = new[new._merge == "left_only"].drop(columns="_merge")
    new["sim_name"] = np.nan; new["sim_addr"] = np.nan; new["from_tfidf"] = np.int8(0)
    P = pd.concat([P, new[P.columns]], ignore_index=True)
    for t in KEY_TYPES:
        P[f"k_{t}"] = P[f"k_{t}"].fillna(0).astype(np.int8)
    P["k_addr_tok_n"] = P["k_addr_tok_n"].fillna(0).astype(np.int16)
    P["k_n"] = P["k_n"].fillna(0).astype(np.int16)
    P["a"] = P.a.astype(np.int64); P["b"] = P.b.astype(np.int64)
    return P

# ---------------------------------------------------------------------
# Exactness checks. The data contains planted look-alikes: same name and street, but a different
# house number (712 vs 719) or a different legal form (LLP vs Private Limited). v1 threw legal words away.
# ---------------------------------------------------------------------
LEGAL_MAP = {"prwt": "pvt", "pwt": "pvt", "prvt": "pvt", "lmtd": "ltd", "ltd": "ltd", "lmt": "ltd", "llp": "llp",
             "llc": "llc", "inc": "inc", "ncrprtd": "inc", "crp": "corp", "crprtn": "corp", "cmpny": "co", "c": "co",
             "plc": "plc", "pc": "pc", "pllc": "pllc", "lp": "lp", "srl": "sarl", "eurl": "eurl", "ss": "sas",
             "ssu": "sas", "sa": "sa", "gmbh": "gmbh", "snc": "snc", "sc": "sci", "sk": "sci", "dba": "dba"}

def _legal_rows(rows):
    out = []
    for nb, an in rows:
        lg = set(LEGAL_MAP.get(skel_tok(t)) for t in nb.split()) - {None}
        if "pvt" in lg: lg.discard("ltd")                  # "Private Limited" is one legal form
        nums = [x.lstrip("0") or "0" for x in re.findall(r"\d+", an)]
        out.append(("|".join(sorted(lg)), " ".join(nums)))
    return out

def add_extra(D):
    """Adds per-record columns: legal (legal form, e.g. 'pvt', 'llp|co') and nums (numbers in the address)."""
    rows = list(zip(D.name_b.tolist(), D.addr_n.tolist())); step = 100_000
    with Pool(N_CPU) as pool:
        res = [r for ch in pool.map(_legal_rows, [rows[i:i + step] for i in range(0, len(rows), step)]) for r in ch]
    D["legal"] = [r[0] for r in res]; D["nums"] = [r[1] for r in res]
    return D

def _near(x, y):
    """Two different numbers that look like a planted change: close in value, or same length and one digit apart."""
    if x == y: return False
    if abs(int(x[:9]) - int(y[:9])) <= 20: return True
    return len(x) == len(y) and sum(p != q for p, q in zip(x, y)) == 1

EXACT_FEATS = ["legal_rel", "legal_s1", "legal_cand", "num_eq", "num_cand_extra", "num_s1_extra", "num_first_eq",
               "num_near", "x_noextra", "x_legal_same"]

def _exact_rows(args):
    out = np.zeros((len(args[0]), 8), np.int16)
    for i, (x, y, p, q) in enumerate(zip(*args)):
        lr = -1 if (not x or not y) else int(x == y)
        pl_, ql = p.split(), q.split(); ps, qs = set(pl_), set(ql)
        ea, eb = ps - qs, qs - ps
        first = -1 if (not pl_ or not ql) else int(pl_[0] == ql[0])
        near = int(any(_near(u, v) for u in ea for v in eb)) if ea and eb else 0
        out[i] = (lr, int(bool(x)), int(bool(y)), int(ps == qs), len(eb), len(ea), first, near)
    return out

def exact_features(P, A, B):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    cols = [A.legal.to_numpy()[a], B.legal.to_numpy()[b], A.nums.to_numpy()[a], B.nums.to_numpy()[b]]
    step = 200_000
    with Pool(N_CPU) as pool:
        res = pool.map(_exact_rows, [[c[i:i + step].tolist() for c in cols] for i in range(0, len(P), step)])
    F = pd.DataFrame(np.concatenate(res) if res else np.zeros((0, 8), np.int16), columns=EXACT_FEATS[:8])
    F["x_noextra"] = ((F.num_cand_extra.to_numpy() == 0) & (pd.Series(cols[3]).str.len().to_numpy() > 0)).astype(np.int8)
    F["x_legal_same"] = (F.legal_rel.to_numpy() == 1).astype(np.int8)
    return F


In [ ]:
# =====================================================================
# Version 10 cleaning fixes (aimed at the "shortlist never found it" mistakes).
#  1. Indian-script words -> Latin with v5's hand-written rules (all 9 scripts share one Unicode layout),
#     then a word list learned from the TRAINING pairs fixes the spelling ('praivet' -> 'private').
#  2. States: v1 missed short names (Ohio, Iowa, Utah, Goa), the code 'TG' (Telangana), and state names
#     written in Indian scripts or as 'Keralam'. Unknown address parts get their state from a list learned from
#     the training pairs (only parts that nearly always mean one state).
#  3. Digit-for-letter typos in words: '5uryaiya' -> 'suryaiya', 'Leam1ng' -> 'leamlng' (v1 only did 0 and 1).
# The Indian-script rules below are copied from Version 5 (v5/v5_core.py).
# =====================================================================
import collections
# ---------------------------------------------------------------------
# 2a. Rule-based transliteration for the 9 Indian scripts.
#     All 9 Unicode blocks share one layout (same letter = same offset from the block start),
#     so one table covers Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu, Kannada, Malayalam.
#     Output uses English-style spelling because Source 1 names are English words in Latin letters.
# ---------------------------------------------------------------------
CONS = {0x15:"k",0x16:"kh",0x17:"g",0x18:"gh",0x19:"n",0x1A:"ch",0x1B:"chh",0x1C:"j",0x1D:"jh",0x1E:"n",
        0x1F:"t",0x20:"th",0x21:"d",0x22:"dh",0x23:"n",0x24:"t",0x25:"th",0x26:"d",0x27:"dh",0x28:"n",0x29:"n",
        0x2A:"p",0x2B:"ph",0x2C:"b",0x2D:"bh",0x2E:"m",0x2F:"y",0x30:"r",0x31:"r",0x32:"l",0x33:"l",0x34:"l",
        0x35:"v",0x36:"sh",0x37:"sh",0x38:"s",0x39:"h",
        0x58:"q",0x59:"kh",0x5A:"gh",0x5B:"z",0x5C:"r",0x5D:"rh",0x5E:"f",0x5F:"y"}
VOW = {0x05:"a",0x06:"a",0x07:"i",0x08:"i",0x09:"u",0x0A:"u",0x0B:"ri",0x0C:"li",0x0D:"e",0x0E:"e",0x0F:"e",
       0x10:"ai",0x11:"o",0x12:"o",0x13:"o",0x14:"au",0x60:"ri",0x61:"li",0x72:"e",0x73:"u",0x74:"u"}
SIGN = {0x3E:"a",0x3F:"i",0x40:"i",0x41:"u",0x42:"u",0x43:"ri",0x44:"ri",0x45:"e",0x46:"e",0x47:"e",0x48:"ai",
        0x49:"o",0x4A:"o",0x4B:"o",0x4C:"au",0x62:"li",0x63:"li",0x55:"",0x56:"ai",0x57:"au"}
NUKTA_MAP = {"k":"q","g":"gh","j":"z","d":"r","dh":"rh","ph":"f"}
CHILLU = {0x0D7A:"n",0x0D7B:"n",0x0D7C:"r",0x0D7D:"l",0x0D7E:"l",0x0D7F:"k",0x0D54:"m",0x0D55:"y",0x0D56:"l"}
_INDIC = re.compile(r"[ऀ-ൿ]")

def indic_word(w):
    """One word in an Indian script -> Latin letters (Latin characters pass through)."""
    w = w.replace("റ്റ", "ട").replace("ന്റ", "ന്ട")  # Malayalam tta/nta
    out, pend = [], None
    def flush(v):
        nonlocal pend
        if pend is not None:
            out.append(pend + (v if v is not None else "\x00"))   # \x00 = inherent 'a', decided below
            pend = None
    for ch in w:
        o = ord(ch)
        if o in CHILLU:
            flush(None); out.append(CHILLU[o]); continue
        if not (0x0900 <= o < 0x0D80):
            flush(None)
            if ch not in "‌‍":
                out.append(unidecode(ch))
            continue
        b = o & ~0x7F; k = o - b
        if k in CONS:
            flush(None); pend = CONS[k]
        elif b == 0x0980 and k == 0x4E:          # Bengali khanda ta
            flush(None); out.append("t")
        elif k == 0x3C:                          # nukta: ka->qa, ja->za, pha->fa ...
            if pend is not None: pend = NUKTA_MAP.get(pend, pend)
        elif k in SIGN:
            flush(SIGN[k])
        elif k == 0x4D:                          # virama: no vowel
            flush("")
        elif k in VOW:
            flush(None); out.append(VOW[k])
        elif k in (0x01, 0x02) or (b == 0x0A00 and k == 0x70):
            flush(None); out.append("\x01")      # nasal sign
        elif k == 0x03:
            flush(None); out.append("h")
        elif 0x66 <= k <= 0x6F:
            flush(None); out.append(str(k - 0x66))
        elif k == 0x50:
            flush(None); out.append("om")
        else:
            flush(None)
    flush(None)
    s = "".join(out)
    s = re.sub(r"\x00$", "", s)                  # word-final inherent vowel is silent: राम -> ram
    s = s.replace("\x00", "a")
    s = re.sub(r"\x01(?=[pbm])", "m", s).replace("\x01", "n")
    return s.lower()

# Filled by learn_dicts() from the training pairs (Source 1 English name <-> Source 2/3 Indian-script name).
NAME_DICT = {}      # transliterated word -> English word, e.g. "praivet" -> "private", "pra." -> "pvt"
SEG_STATE = {}      # address part (lower case) -> state code, learned from training pairs

def to_latin(s, use_dict=True):
    """Indian-script words -> Latin (learned word list first, then rules); everything else -> unidecode."""
    s = str(s)
    if _INDIC.search(s):
        ws = []
        for w in s.split():
            if _INDIC.search(w):
                t = indic_word(w)
                ws.append(NAME_DICT.get(t, t) if use_dict else t)
            else:
                ws.append(w)
        s = " ".join(ws)
    return unidecode(s).lower()

def base(s):
    s = to_latin(s)
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

_LEET = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "7": "t", "8": "b"})
def skel_tok(t):
    """v1 skeleton, plus: a word of 4+ characters with letters and ONE digit gets the digit read as a letter."""
    if not t or t.isdigit():
        return t
    if re.search(r"[a-z]", t):
        nd = sum(ch.isdigit() for ch in t)
        t = t.translate(_LEET) if (nd == 1 and len(t) >= 4) else t.replace("0", "o").replace("1", "l")
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

_EXACT = {full: code for d in (US, IN) for full, code in d.items()}
_EXACT.update({"tg": "ts", "keralam": "kl", "orissa": "od", "pondicherry": "py"})
_states_v1 = states_of
def _seg_key(seg):
    return re.sub(r"\s+", " ", str(seg).strip().lower())

def states_of(raw_addr):
    """v1 state reading + whole-part exact names (Ohio ...) + learned parts (Indian scripts, 'Keralam' ...)."""
    found = set(x for x in _states_v1(raw_addr).split("|") if x)
    for seg in str(raw_addr).split(","):
        w = re.sub(r"[^a-z ]", " ", unidecode(seg).lower()).strip()
        w = re.sub(r"\s+", " ", w)
        if w in _EXACT:
            found.add(MERGE_ST.get(_EXACT[w], _EXACT[w]))
    if not found:
        for seg in str(raw_addr).split(","):
            c = SEG_STATE.get(_seg_key(seg))
            if c: found.add(c)
    return "|".join(sorted(found))

def _no_state(addrs):
    return np.array([_states_v1(a) == "" for a in addrs], bool)

def _pool_map(fn, items, step=200_000):
    with Pool(N_CPU) as pool:
        return pool.map(fn, [items[i:i + step] for i in range(0, len(items), step)])

def learn_dicts(s1, gt, others, min_n=5):
    """Learns NAME_DICT (Indian-script word -> English word) and SEG_STATE (address part -> state) from the
    TRAINING pairs only. s1: raw train Source 1, gt: ground truth, others: raw Source 2/3 frames."""
    link = {}
    for s, m in zip(gt.source1_entity_id, gt.matched_entity_ids):
        for c in m.split(","):
            if c: link[c] = s
    s1n = dict(zip(s1.entity_id, s1.business_name)); s1a = dict(zip(s1.entity_id, s1.business_address))
    wc = collections.defaultdict(collections.Counter); sc = collections.defaultdict(collections.Counter)
    for B in others:
        for eid, n, a in zip(B.entity_id, B.business_name, B.business_address):
            s = link.get(eid)
            if s is None: continue
            if _INDIC.search(n):
                ta = n.split(); tb = re.sub(r"[^a-z0-9 ]", " ", unidecode(s1n[s]).lower()).split()
                if len(ta) == len(tb):
                    for x, y in zip(ta, tb):
                        if _INDIC.search(x): wc[indic_word(x)][y] += 1
            if a.strip() and not _states_v1(a):
                st = [x for x in states_of(s1a[s]).split("|") if x]
                if len(st) == 1:
                    for seg in a.split(","):
                        k = _seg_key(seg)
                        if k and not re.search(r"\d", k): sc[k][st[0]] += 1
    nd = {k: c.most_common(1)[0][0] for k, c in wc.items()
          if c.most_common(1)[0][1] >= 3 and c.most_common(1)[0][1] / sum(c.values()) >= 0.6}
    sd = {k: c.most_common(1)[0][0] for k, c in sc.items()
          if c.most_common(1)[0][1] >= min_n and c.most_common(1)[0][1] / sum(c.values()) >= 0.95}
    return nd, sd


In [ ]:
# =====================================================================
# Version 10 fixes: new pair features aimed at the mistake types found in the error analysis.
# (See v10/error_types.md for the table of mistake types and what each fix bought.)
# =====================================================================
from rapidfuzz.distance import Levenshtein as _Lev

def _initials(words):
    return "".join(w[0] for w in words if w and not w.isdigit())

def _tok_rows(args):
    """Word-by-word comparison of two names (legal words removed).
    Catches planted look-alikes where exactly ONE word was changed ('Chandrraj' -> 'Chandrreon'),
    and short/website names made of initials ('mc.com' for 'Marquez, Conway & ...')."""
    xs, ys = args
    out = np.zeros((len(xs), 8), np.float32)
    for i, (x, y) in enumerate(zip(xs, ys)):
        a, b = x.split(), y.split()
        if not a or not b:
            out[i] = (-1, -1, -1, -1, -1, 0, 0, -1); continue
        # best partner of each word on the other side (normalised edit similarity, 0..1)
        def best(u, V): return max(JaroWinkler.normalized_similarity(u, v) for v in V)
        ba = [best(u, b) for u in a]; bb = [best(v, a) for v in b]
        miss_a = sum(s < 0.85 for s in ba); miss_b = sum(s < 0.85 for s in bb)
        near_a = [s for s in ba if 0.85 <= s < 1.0]              # words that are close but not equal (a changed word)
        worst = min(ba)
        one_sub = int(len(a) == len(b) and miss_a == 0 and len(near_a) == 1 and min(bb) < 1.0)
        ia, ib = _initials(a), _initials(b)
        yb = "".join(b)
        acr = int(len(yb) <= 5 and len(a) >= 2 and (yb == ia or yb.startswith(ia) or ia.startswith(yb) and len(yb) >= 2))
        acr_rev = int(len(a) == 1 and len(b) >= 2 and ("".join(a) == ib or ib.startswith("".join(a)) and len(a[0]) >= 2
                                                     and len(a[0]) <= 5))
        out[i] = (miss_a, miss_b, worst, min(bb), len(near_a), one_sub, acr, max(acr, acr_rev))
    return out

TOK_FEATS = ["w_miss_s1", "w_miss_c", "w_worst_s1", "w_worst_c", "w_near_n", "w_one_sub", "acr", "acr_any"]

def token_features(P, A, B, F):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    xs, ys = A.name_w.to_numpy()[a], B.name_w.to_numpy()[b]
    step = 200_000
    with Pool(N_CPU) as pool:
        res = pool.map(_tok_rows, [(xs[i:i + step].tolist(), ys[i:i + step].tolist()) for i in range(0, len(a), step)])
    R = np.concatenate(res) if res else np.zeros((0, len(TOK_FEATS)), np.float32)
    for j, c in enumerate(TOK_FEATS):
        F[c] = R[:, j]
    return F

def features_v10(P, A, B, F):
    F = token_features(P, A, B, F)
    return F

# ---------------------------------------------------------------------
# Stage 2 context features, copied from Version 2 (v2/v23_code.py): for each candidate, how strong the
# business's other candidates are, and whether this record looks like the other strong copies (name + address).
# ---------------------------------------------------------------------
K_CTX = 8            # each candidate is compared with this many of the business's best candidates
S2_EXTRA = ["p1", "p1_rank", "p1_max", "p1_2nd", "p1_sum", "p1_rel", "n_hi", "n_kept",
            "sup_max", "sup_sum", "sup_cnt", "sup_name_max", "sup_addr_max", "sup_other_src"]

def stage2_features(key, p1, cname, caddr, is_s3, K=K_CTX):
    """Context features. key: business of each row; p1: stage-1 chance; cname/caddr: candidate's cleaned text."""
    d = pd.DataFrame({"key": key, "p1": p1.astype(np.float32), "row": np.arange(len(key)), "s3": is_s3})
    d = d.sort_values(["key", "p1"], ascending=[True, False], kind="stable")
    g = d.groupby("key", sort=False)
    d["p1_rank"] = (g.cumcount() + 1).astype(np.float32)
    d["p1_max"] = g.p1.transform("max")
    d["p1_2nd"] = d.key.map(d[d.p1_rank == 2].set_index("key").p1).fillna(0).to_numpy(np.float32)
    d["p1_sum"] = g.p1.transform("sum")
    d["p1_rel"] = d.p1 / d.p1_max.clip(lower=1e-6)
    d["n_hi"] = (d.p1 > 0.5).groupby(d.key).transform("sum").astype(np.float32)
    d["n_kept"] = g.p1.transform("size").astype(np.float32)
    top = d[d.p1_rank <= K][["key", "row", "p1", "s3"]]
    j = top.merge(top, on="key", suffixes=("", "_o"))
    j = j[j.row != j.row_o]
    r, ro = j.row.to_numpy(), j.row_o.to_numpy()
    sn = cpdist(cname[r].tolist(), cname[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    sa = cpdist(caddr[r].tolist(), caddr[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    po = j.p1_o.to_numpy()
    s = (sn + sa) / 2
    j = pd.DataFrame({"row": r, "w": po * s, "strong": ((s > 0.85) & (po > 0.5)).astype(np.int8),
                      "wn": po * sn, "wa": po * sa, "wx": np.where(j.s3.to_numpy() != j.s3_o.to_numpy(), po * s, 0)})
    agg = j.groupby("row").agg(sup_max=("w", "max"), sup_sum=("w", "sum"), sup_cnt=("strong", "sum"),
                               sup_name_max=("wn", "max"), sup_addr_max=("wa", "max"), sup_other_src=("wx", "max"))
    d = d.set_index("row").sort_index().join(agg)
    return d[S2_EXTRA].fillna(0).astype(np.float32).reset_index(drop=True)


# stage-1 pair features that stage 2 also sees
S2_KEEP = ["cand_addr_len", "s1_addr_len", "name_exact", "k_name", "k_nameset", "k_name_cb", "k_name_ca", "num_cand_extra",
           "num_s1_extra", "num_near", "legal_rel", "cand_nonascii", "is_s3", "nb_tset", "ask_tset", "from_tfidf"]
S2_PARAMS = dict(learning_rate=0.03, num_leaves=63, min_child_samples=100, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 150_000))   # training businesses (same 150k as v1/v7/v8, so scores compare)
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)
K_UNK = 10              # candidates with no readable state: how many name look-alikes to take (v1 used 10)
S1_MAX_TREES = 600      # stage-1 model size limit (more trees = slower test prediction; Kaggle stops at 12 hours)
KEEP_P = 0.01           # stage 1 -> stage 2: keep candidates with at least this chance ...
KTOP = 20               # ... and at most this many per business and source. This is also candidate_pairs.tsv.
SMOKE = 0             # 1 = tiny quick run on a slice of the data (checks the code only)
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU, "| training businesses:", N_TRAIN_S1, "| smoke test:", bool(SMOKE))


In [ ]:
# =====================================================================
# Version 10, step 1: build the training table.
# Shortlist = v1 TF-IDF shortlist + v7 key pairs. Features = v1 + v8 (rarity, sound codes, ranks)
# + v7 (key hits, house-number and legal-form checks) + v10 fixes (added in features_v10).
# Every block is saved to disk (WORK/v10_parts) so memory stays low and a re-run skips finished blocks.
# Also saves the raw text of the held-out businesses and their candidates, for the error analysis.
# =====================================================================
PARTS = f"{WORK}/v10_parts"; os.makedirs(PARTS, exist_ok=True)
T0 = time.time()
log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
# v10 cleaning: learn the Indian-script word list and the address-part -> state list from the training pairs
DICTS = f"{WORK}/v10_dicts.json"
if not os.path.exists(DICTS):
    parts = []
    for src in ("train_source2", "train_source3"):
        B = load(ROOT, src)
        keep = B.business_name.str.contains(_INDIC.pattern, regex=True).to_numpy() | \
               np.concatenate(_pool_map(_no_state, B.business_address.tolist()))
        parts.append(B[keep].reset_index(drop=True)); del B; gc.collect()
    nd, sd = learn_dicts(s1, gt, parts); del parts; gc.collect()
    json.dump({"name_dict": nd, "seg_state": sd}, open(DICTS, "w"), ensure_ascii=False)
D = json.load(open(DICTS)); NAME_DICT.update(D["name_dict"]); SEG_STATE.update(D["seg_state"])
log("learned words:", len(NAME_DICT), "| learned address parts -> state:", len(SEG_STATE))
del gt; gc.collect()
if SMOKE: s1 = s1.sample(60_000, random_state=5).reset_index(drop=True)
samp = s1.sample(min(4000 if SMOKE else N_TRAIN_S1, len(s1)), random_state=0).index.to_numpy()   # same 150k as v1/v7/v8
RAW1 = s1.iloc[samp][["entity_id", "business_name", "business_address", "country"]].reset_index(drop=True)
S1 = add_extra(prepare_extra(prepare(s1))); del s1; gc.collect(); log("prepared all train S1", S1.shape)
KA = make_keys(S1).filter(pl.col("row").is_in(samp)); log("S1 keys", KA.height)
A = S1.iloc[samp].reset_index(drop=True); del S1; free_mem()
rng = np.random.default_rng(42)
val_ids = set(rng.choice(A.entity_id.to_numpy(), int(0.2 * len(A)), replace=False))              # same 30k as v1
json.dump({"vt": {s: sorted(truth[s]) for s in val_ids}, "n_s1": len(A),
           "all_true": sum(len(truth[s]) for s in A.entity_id)}, open(f"{PARTS}/val.json", "w"))
A.assign(business_name=RAW1.business_name.to_numpy(), business_address=RAW1.business_address.to_numpy()) \
    .to_parquet(f"{PARTS}/text_s1.parquet")                                        # cleaned + raw text, for analysis
want_val = set().union(*(truth[s] for s in val_ids))

for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    if os.path.exists(f"{PARTS}/{src}.done"):
        log(src, "already done"); continue
    log("loading", src); B = load(ROOT, src)
    if SMOKE:
        want = set().union(*(truth[s] for s in A.entity_id))
        B = pd.concat([B[B.entity_id.isin(want)], B.sample(200_000, random_state=0)]).drop_duplicates("entity_id")
        B = B.reset_index(drop=True)
    RAWB = B[["entity_id", "business_name", "business_address", "country"]].copy()
    B = add_extra(prepare_extra(prepare(B))); log("prepared", src, B.shape)
    keep_rows = set()
    for country in A.country.unique():
        sel = (A.country == country).to_numpy()
        Ac = A[sel].reset_index(drop=True); Bc = B[B.country == country].reset_index(drop=True)
        if len(Ac) == 0 or len(Bc) == 0: continue
        P = Index(Bc, k_unk=K_UNK).query(Ac)
        KB = make_keys(Bc)
        P = merge_key_pairs(P, key_pairs(KA, KB, samp[sel])); del KB
        R = Rarity(Bc); F = features_v8(P, Ac, Bc, is_s3, R); del R
        for c in KEY_FEATS + ["from_tfidf"]:
            F[c] = P[c].to_numpy()
        E = exact_features(P, Ac, Bc)
        for c in EXACT_FEATS:
            F[c] = E[c].to_numpy()
        del E
        F = features_v10(P, Ac, Bc, F)
        m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a], "cand_id": Bc.entity_id.to_numpy()[P.b]})
        m["label"] = np.array([c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)], np.int8)
        m["src"] = np.int8(is_s3); m["country"] = country
        keep_rows |= set(m.cand_id)
        F.astype(np.float32).to_parquet(f"{PARTS}/X_{src}_{country}.parquet")
        m.to_parquet(f"{PARTS}/M_{src}_{country}.parquet")
        log(src, country, "pairs", len(m), "true", int(m.label.sum()), "| features", F.shape[1],
            "| minutes", round((time.time() - T0) / 60, 1))
        del Ac, Bc, P, F, m; free_mem()
    keep_rows |= want_val
    k = B.entity_id.isin(keep_rows).to_numpy()
    B[k].assign(business_name=RAWB.business_name.to_numpy()[k], business_address=RAWB.business_address.to_numpy()[k]) \
        .to_parquet(f"{PARTS}/text_{src}.parquet")                                  # text of every candidate, for analysis
    open(f"{PARTS}/{src}.done", "w").close()
    del B, RAWB; free_mem()
del KA, A; free_mem()
log("training table built | minutes", round((time.time() - T0) / 60, 1))


In [ ]:
# =====================================================================
# Version 10, step 2: train.
#  Stage 1: one LightGBM scores every pair on its own (v8's best single-model settings).
#           Its number of trees is picked on the 30k held-out businesses (same ones as v1/v7/v8).
#           Training businesses also get "out-of-fold" scores (2 halves: each half scored by a model that never saw it),
#           so stage 2 learns from honest scores, like the ones it will see on the test set.
#  Stage 2: v2's idea. Looks at all candidates of a business together ("does this record look like the
#           business's other strong copies?") and gives the final chance. Only pairs with stage-1 chance >= KEEP_P
#           (top KTOP per business) go on: this is also the small candidate list the organisers now ask for.
# =====================================================================
T0 = time.time()
V = json.load(open(f"{PARTS}/val.json")); vt = {s: set(v) for s, v in V["vt"].items()}; vl = list(vt)
blocks = sorted(glob.glob(f"{PARTS}/X_*.parquet"))
M = pd.concat([pd.read_parquet(f.replace("/X_", "/M_")) for f in blocks], ignore_index=True)
iv0 = M.s1_id.isin(vt).to_numpy(); perm = np.r_[np.flatnonzero(~iv0), np.flatnonzero(iv0)]; n_tr = int((~iv0).sum())
dest = np.empty(len(perm), np.int64); dest[perm] = np.arange(len(perm))
FEATS = list(pd.read_parquet(blocks[0]).columns)
X = np.lib.format.open_memmap(f"{WORK}/v10_X.npy", "w+", np.float32, (len(M), len(FEATS)))   # on disk: training rows first
o = 0
for f in blocks:
    F = pd.read_parquet(f); X[dest[o:o + len(F)]] = F[FEATS].to_numpy(np.float32); o += len(F); del F; gc.collect()
    os.remove(f)                                    # free disk space (the table now lives in v10_X.npy)
X.flush(); del X, dest; gc.collect()
X = np.load(f"{WORK}/v10_X.npy", mmap_mode="r")
M = M.iloc[perm].reset_index(drop=True); y = M.label.to_numpy(); del perm, iv0; gc.collect()
log("pairs", len(M), "| features", len(FEATS), "| shortlist recall", round(y.sum() / V["all_true"], 4))
log("baseline all-empty", round(macro_f05({}, vt, vl), 4), "| ceiling (perfect model on this shortlist)",
    round(macro_f05(M[n_tr:][y[n_tr:] == 1].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
GRID = np.round(np.arange(0.30, 0.901, 0.025), 3)

def best_cut(pv, label):
    res = [(t, macro_f05(decide(pv, t), vt, vl)) for t in GRID]
    t, s = max(res, key=lambda r: r[1])
    log(f"BEST val F0.5 [{label}] = {s:.4f} at cut-off {t}")
    return float(t), float(s)

# ---- stage 1 ----
S1_PARAMS = dict(learning_rate=0.1, num_leaves=127, min_child_samples=50, subsample=0.7, subsample_freq=1,
                 colsample_bytree=0.5, reg_lambda=5.0, random_state=1, n_jobs=N_CPU, verbose=-1)
p1 = np.zeros(len(M), np.float32)
m1 = lgb.LGBMClassifier(n_estimators=S1_MAX_TREES, **S1_PARAMS)
m1.fit(X[:n_tr], y[:n_tr], eval_set=[(X[n_tr:], y[n_tr:])], callbacks=[lgb.early_stopping(100, verbose=False)])
NT = m1.best_iteration_; p1[n_tr:] = m1.predict_proba(X[n_tr:])[:, 1]
m1.booster_.save_model(f"{WORK}/v10_stage1.txt", num_iteration=NT)
log("stage 1 trees", NT, "| minutes", round((time.time() - T0) / 60, 1))
R1 = best_cut(M[n_tr:][["s1_id", "cand_id"]].assign(p=p1[n_tr:]), "stage 1 alone")
imp = pd.Series(m1.booster_.feature_importance("gain"), FEATS); del m1; gc.collect()
log("top stage-1 features:\n" + (imp / imp.sum()).sort_values(ascending=False).round(4).head(15).to_string())
fold = (pd.util.hash_pandas_object(M.s1_id[:n_tr], index=False).to_numpy() % 2).astype(np.int8)
for k in (0, 1):                    # out-of-fold scores for the training businesses
    tr = np.flatnonzero(fold == k); te = np.flatnonzero(fold != k)
    m = lgb.LGBMClassifier(n_estimators=NT, **S1_PARAMS); m.fit(X[tr], y[tr])
    p1[te] = m.predict_proba(X[te])[:, 1]; del m; gc.collect()
    log("out-of-fold half", k + 1, "of 2 done | minutes", round((time.time() - T0) / 60, 1))

# ---- the small candidate list: how many true matches does each size keep? ----
M["p1"] = p1; M["rk"] = M.groupby("s1_id").p1.rank(ascending=False, method="first").astype(np.float32)
isv = np.arange(len(M)) >= n_tr; n_true_val = sum(len(t) for t in vt.values())
for k in (5, 10, 15, 20):
    kp = (M.p1 >= KEEP_P) & (M.rk <= k) & isv
    log(f"candidate list p1>={KEEP_P}, top {k}: {kp.sum() / len(vl):.2f} per business, keeps "
        f"{M.label[kp].sum() / n_true_val:.4f} of all true matches (full shortlist {M.label[isv].sum() / n_true_val:.4f})")
keep = ((M.p1 >= KEEP_P) & (M.rk <= KTOP)).to_numpy()
S = M[keep].reset_index(drop=True)
F2 = pd.DataFrame(np.asarray(X[np.flatnonzero(keep)][:, [FEATS.index(c) for c in S2_KEEP]]), columns=S2_KEEP)
del X, M, p1; gc.collect(); os.remove(f"{WORK}/v10_X.npy")

# ---- stage 2 ----
T = pd.concat([pd.read_parquet(f"{PARTS}/text_train_source{k}.parquet", columns=["entity_id", "name_sk", "addr_sk"])
               for k in (2, 3)]).drop_duplicates("entity_id").set_index("entity_id").loc[S.cand_id]
G = stage2_features(S.s1_id.to_numpy(), S.p1.to_numpy(), T.name_sk.to_numpy(), T.addr_sk.to_numpy(), S.src.to_numpy())
del T; gc.collect()
X2 = pd.concat([G, F2], axis=1).astype(np.float32); y2 = S.label.to_numpy()
v = S.s1_id.isin(vt).to_numpy()
m2 = lgb.LGBMClassifier(n_estimators=3000, **S2_PARAMS)
m2.fit(X2[~v], y2[~v], eval_set=[(X2[v], y2[v])], callbacks=[lgb.early_stopping(100, verbose=False)])
p2 = m2.predict_proba(X2[v])[:, 1].astype(np.float32)
m2.booster_.save_model(f"{WORK}/v10_stage2.txt", num_iteration=m2.best_iteration_)
PV = S[v][["s1_id", "cand_id"]].assign(p=p2)
CUT, BEST = best_cut(PV, "stage 2 = FINAL")
PV[PV.p >= 0.05].reset_index(drop=True).to_parquet(f"{WORK}/val_scores.parquet")        # for the final blend (v14)
json.dump({"stage1_features": FEATS, "stage1_trees": NT, "stage2_features": list(X2.columns), "threshold": CUT,
           "val_f05": BEST, "val_f05_stage1": R1[1], "keep_p": KEEP_P, "ktop": KTOP},
          open(f"{WORK}/v10_meta.json", "w"), indent=1)
json.dump({"cut": CUT, "val_f05": BEST}, open(f"{WORK}/val_scores.json", "w"))
log(f"FINAL: val F0.5 {BEST:.4f} at cut-off {CUT} (v1 0.9529, v8 0.9673) | training minutes", round((time.time() - T0) / 60, 1))
del S, F2, G, X2, m2; gc.collect()


In [ ]:
# =====================================================================
# Version 10, step 3: predict the test set, in batches saved to disk (safe to resume), then write both TSVs.
#  Pass 1 (per source, per batch): shortlist + key pairs, features, stage-1 chance. Only candidates with
#          chance >= KEEP_P (top KTOP per business and source) are saved, with the few columns stage 2 needs.
#  Pass 2 (per batch, both sources together): stage-2 chance, then the decision.
# =====================================================================
T0 = time.time()
meta = json.load(open(f"{WORK}/v10_meta.json"))
B1 = lgb.Booster(model_file=f"{WORK}/v10_stage1.txt"); B2 = lgb.Booster(model_file=f"{WORK}/v10_stage2.txt")
F1, F2N = meta["stage1_features"], meta["stage2_features"]
log("stage-1 features", len(F1), "| stage-2 features", len(F2N), "| cut-off", meta["threshold"])
PP = f"{WORK}/parts_v10"; os.makedirs(PP, exist_ok=True)
log("loading test Source 1"); A_full = load(ROOT, "test_source1")
s1_ids = A_full.entity_id.to_numpy()
A = A_full.groupby("country").head(1500).reset_index(drop=True) if SMOKE else A_full
del A_full; gc.collect()
A = add_extra(prepare_extra(prepare(A))); log("prepared", A.shape)
print(A.country.value_counts().to_string())
KA = make_keys(A); log("test S1 keys", KA.height)       # keys of ALL test businesses, so 'how many share a key' is right
row_of = {s: i for i, s in enumerate(s1_ids)}
A_rowid = np.array([row_of[s] for s in A.entity_id], np.int32)
countries = list(A.country.unique())
jobs = {c: list(range(0, int((A.country == c).sum()), BATCH)) for c in countries}

for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in countries for i in jobs[c] if not os.path.exists(f"{PP}/{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    log("loading", src); B = load(ROOT, src)
    if SMOKE: B = B.sample(300_000, random_state=0).reset_index(drop=True)
    B = add_extra(prepare_extra(prepare(B))); log("prepared", src, B.shape)
    for country in countries:                 # any country label works, France included
        if not any(c == country for c, _ in todo): continue
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        idx = Index(Bc, k_unk=K_UNK) if len(Bc) else None
        R = Rarity(Bc) if len(Bc) else None
        KB = make_keys(Bc) if len(Bc) else None
        bids = Bc.entity_id.to_numpy()
        for i in jobs[country]:
            out = f"{PP}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = idx.query(Ac) if idx is not None else pd.DataFrame({"a": [], "b": []})
            if len(P):
                P = merge_key_pairs(P, key_pairs(KA, KB, rows))
                F = features_v8(P, Ac, Bc, is_s3, R)
                for c in KEY_FEATS + ["from_tfidf"]: F[c] = P[c].to_numpy()
                E = exact_features(P, Ac, Bc)
                for c in EXACT_FEATS: F[c] = E[c].to_numpy()
                del E
                F = features_v10(P, Ac, Bc, F)
                p1 = B1.predict(F[F1].to_numpy(np.float32), num_threads=N_CPU).astype(np.float32)
                D = pd.DataFrame({"s1_row": A_rowid[rows[P.a.to_numpy()]], "cand_id": bids[P.b.to_numpy()], "p1": p1,
                                  "src": np.int8(is_s3), "cname": Bc.name_sk.to_numpy()[P.b.to_numpy()],
                                  "caddr": Bc.addr_sk.to_numpy()[P.b.to_numpy()]})
                for c in S2_KEEP: D[c] = F[c].to_numpy(np.float32)
                n_all = len(D); del F
                D = D[D.p1 >= KEEP_P]
                D = D[D.groupby("s1_row").p1.rank(ascending=False, method="first") <= KTOP]
            else:
                D = pd.DataFrame(columns=["s1_row", "cand_id", "p1", "src", "cname", "caddr"] + S2_KEEP); n_all = 0
            D.reset_index(drop=True).to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{len(jobs[country])}", "shortlist", n_all, "kept", len(D),
                "| elapsed min", round((time.time() - T0) / 60, 1))
            del P, D; gc.collect()
        del idx, R, KB, Bc; free_mem()
    del B; free_mem()
del KA; free_mem()

# ---- pass 2: stage 2 per batch (both sources together), then the decision ----
log("stage 2")
scored = []
for c in countries:
    for i in jobs[c]:
        out = f"{PP}/s2_{c}_{i}.parquet"
        if not os.path.exists(out):
            D = pd.concat([pd.read_parquet(f"{PP}/{s}_{c}_{i}.parquet") for s in ("test_source2", "test_source3")],
                          ignore_index=True)
            if len(D):
                G = stage2_features(D.s1_row.to_numpy(), D.p1.to_numpy(np.float32), D.cname.to_numpy(),
                                    D.caddr.to_numpy(), D.src.to_numpy())
                X2 = pd.concat([G, D[S2_KEEP].reset_index(drop=True)], axis=1)[F2N].to_numpy(np.float32)
                D["p"] = B2.predict(X2, num_threads=N_CPU).astype(np.float32); del G, X2
            else:
                D["p"] = np.zeros(0, np.float32)
            D[["s1_row", "cand_id", "p"]].to_parquet(out); del D; gc.collect()
        scored.append(out)
log("stage 2 done | elapsed min", round((time.time() - T0) / 60, 1))
S = pl.concat([pl.read_parquet(f) for f in scored]).with_columns(pl.col("s1_row").cast(pl.Int32))
ids = pl.DataFrame({"s1_row": np.arange(len(s1_ids), dtype=np.int32), "s1_id": s1_ids.astype(str)})
S.filter(pl.col("p") >= 0.05).join(ids, on="s1_row").select(["s1_id", "cand_id", "p"]) \
    .write_parquet(f"{WORK}/test_scores.parquet")                                        # for the final blend (v14)
del ids
# each Source 2/3 record goes to its single best business, and only if its chance passes the cut-off
best = S.filter(pl.col("p") >= meta["threshold"]).sort("p", descending=True).unique(subset="cand_id", keep="first",
                                                                                       maintain_order=True)
match = best.group_by("s1_row").agg(pl.col("cand_id").sort())
mline = np.full(len(s1_ids), "", dtype=object)
for r, c in zip(match["s1_row"].to_list(), match["cand_id"].to_list()):
    mline[r] = ",".join(c)
cand = S.group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
cline = np.full(len(s1_ids), "", dtype=object)
for r, c in zip(cand["s1_row"].to_list(), cand["cand_id"].to_list()):
    cline[r] = ",".join(c)
n = int((mline != "").sum()); os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as fh:
    fh.write("source1_entity_id\tmatched_entity_ids\n")
    for s, m in zip(s1_ids, mline): fh.write(s + "\t" + m + "\n")
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as fh:
    fh.write("source1_entity_id\tcandidate_entity_ids\n")
    for s, c in zip(s1_ids, cline): fh.write(s + "\t" + c + "\n")
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(best.height / len(s1_ids), 2), "| candidates per business", round(S.height / len(s1_ids), 2),
    "| predict minutes", round((time.time() - T0) / 60, 1))
del S, best, match, cand, mline, cline; gc.collect()


In [ ]:
# official format check of matching_results.tsv (the candidate file check needs too much memory; the file is still written)
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
import subprocess
print(subprocess.run(['python3', validator, '--matching', f'{WORK}/output/matching_results.tsv', '--test-dir', test_dir,
                      '--candidate', 'none.tsv'], capture_output=True, text=True).stdout[-3000:])

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`** (upload this) and keep `candidate_pairs.tsv` for the final zip. `test_scores.parquet` is for the final blend (Version 14).